# Portfolio risk decomposition, budgeting, and allocation

This notebook covers position-level risk analytics and capital allocation exposed by `finstack_quant.portfolio`:

- **Parametric VaR / ES decomposition** -- Euler (component) contributions, plus marginal and incremental VaR, from a covariance matrix.
- **Historical VaR / ES decomposition** -- the same Euler split driven by a scenario P&L matrix.
- **Risk budgeting** -- comparing each position's component VaR against a target risk share and flagging breaches.
- **Capital allocation** -- inverse-volatility and risk-budget weighting schemes.

These are the computable typed entry points; what-if and factor-stress engines currently expose result deserialization only.

## Imports

In [1]:
import json
import math

import pandas as pd

from finstack_quant.portfolio import (
    parametric_var_decomposition_typed,
    historical_var_decomposition_typed,
    position_component_var,
    evaluate_risk_budget_typed,
    allocate_weights,
    validate_allocation_json,
)

POSITIONS = ["Equity", "Credit", "Rates"]
WEIGHTS = [0.50, 0.30, 0.20]
# Annualised return covariance across the three sleeves
COVARIANCE = [
    [0.0400, 0.0120, -0.0020],
    [0.0120, 0.0200, 0.0010],
    [-0.0020, 0.0010, 0.0064],
]

## Parametric VaR / ES decomposition (Euler)

Under the Euler principle the position **component VaRs sum exactly to the portfolio VaR** (the residual is numerically zero). Each contribution also reports:

- **marginal VaR** -- sensitivity of portfolio VaR to the position weight,
- **incremental VaR** -- the change in portfolio VaR from removing the position (when `compute_incremental=True`).

In [2]:
decomp = parametric_var_decomposition_typed(
    POSITIONS, WEIGHTS, COVARIANCE, confidence=0.99, compute_incremental=True
)

print(f"Portfolio VaR (99%): {decomp.portfolio_var:.4f}    ES: {decomp.portfolio_es:.4f}    method: {decomp.method}")
pd.DataFrame([
    {
        "position": c.position_id,
        "component_var": c.component_var,
        "relative_var": c.relative_var,
        "marginal_var": c.marginal_var,
        "incremental_var": c.incremental_var,
    }
    for c in decomp.var_contributions
])

Portfolio VaR (99%): 0.2885    ES: 0.3305    method: parametric


,position,component_var,relative_var,marginal_var,incremental_var
0,Equity,0.217626,0.754422,0.435252,0.179949
1,Credit,0.068665,0.238033,0.228883,0.057513
2,Rates,0.002176,0.007544,0.010881,-0.000225


In [3]:
total_component = sum(c.component_var for c in decomp.var_contributions)
print(f"sum(component VaR) = {total_component:.6f}")
print(f"portfolio VaR      = {decomp.portfolio_var:.6f}")
print(f"Euler residual     = {decomp.euler_residual:.2e}")
print(f"sum(relative VaR)  = {sum(c.relative_var for c in decomp.var_contributions):.6f}")
print(f"component VaR(Equity) via helper = {position_component_var(decomp, 'Equity'):.6f}")

sum(component VaR) = 0.288467
portfolio VaR      = 0.288467
Euler residual     = 5.55e-17
sum(relative VaR)  = 1.000000
component VaR(Equity) via helper = 0.217626


## Historical VaR / ES decomposition

The historical engine takes a **positions x scenarios** P&L matrix (one row of scenario P&Ls per position) and produces the same Euler split empirically. Marginal / incremental fields are not defined for the historical method (they come back as `None`).

In [4]:
def scenario_pnls(n_scenarios=250):
    # Deterministic synthetic P&L paths; the credit sleeve carries a small left tail.
    equity = [0.020 * math.sin(i / 7.0) - 0.001 for i in range(n_scenarios)]
    credit = [(-0.080 if i < 6 else 0.004 * math.cos(i / 5.0) + 0.001) for i in range(n_scenarios)]
    rates = [0.005 * math.cos(i / 11.0) for i in range(n_scenarios)]
    return [equity, credit, rates]

hist = historical_var_decomposition_typed(POSITIONS, scenario_pnls(), confidence=0.95)

print(f"Historical VaR (95%): {hist.portfolio_var:.4f}    ES: {hist.portfolio_es:.4f}    method: {hist.method}")
pd.DataFrame([
    {"position": c.position_id, "component_var": c.component_var, "relative_var": c.relative_var}
    for c in hist.var_contributions
])

Historical VaR (95%): 0.0232    ES: 0.0469    method: historical


,position,component_var,relative_var
0,Equity,0.003303,0.142228
1,Credit,0.020025,0.862389
2,Rates,-0.000107,-0.004616


## Risk budgeting

`evaluate_risk_budget_typed` compares each position's **actual component VaR** (here taken from the parametric decomposition above) against a **target share** of portfolio VaR. Utilisation above the threshold (1.10 below) marks a breach.

In [5]:
actual_component_var = [c.component_var for c in decomp.var_contributions]
target_shares = [0.50, 0.30, 0.20]

budget = evaluate_risk_budget_typed(
    POSITIONS, actual_component_var, target_shares, decomp.portfolio_var, 1.10
)

print(f"any breach: {budget.has_breach}    total overbudget: {budget.total_overbudget:.4f}")
pd.DataFrame([
    {
        "position": e.position_id,
        "actual_cVaR": e.actual_component_var,
        "target_cVaR": e.target_component_var,
        "utilization": e.utilization,
        "excess": e.excess,
    }
    for e in budget.positions
])

any breach: True    total overbudget: 0.0734


,position,actual_cVaR,target_cVaR,utilization,excess
0,Equity,0.217626,0.144234,1.508845,0.073393
1,Credit,0.068665,0.086540,0.793444,-0.017875
2,Rates,0.002176,0.057693,0.037721,-0.055517


Equity carries far more than its 50% target share of risk (its high variance and positive correlations concentrate component VaR), so it breaches while Credit and Rates sit under budget.

## Capital allocation schemes

`allocate_weights` takes a JSON spec and returns weights and capital per strategy. Two common schemes:

- **inverse_volatility** -- weight inversely to each strategy's realised volatility,
- **risk_budget** -- solve for weights that hit target risk-contribution shares given a covariance matrix.

`validate_allocation_json` checks a spec before running it.

In [6]:
inv_vol_spec = {
    "scheme": "inverse_volatility",
    "total_capital": 1_000_000.0,
    "strategies": [
        {"id": "low_vol_carry", "returns": [0.010, 0.012, 0.009, 0.011, 0.010]},
        {"id": "trend", "returns": [0.050, -0.040, 0.045, -0.035, 0.030]},
        {"id": "value", "returns": [0.020, -0.010, 0.015, 0.005, -0.005]},
    ],
    "money_decimal_places": 2,
}
validate_allocation_json(json.dumps(inv_vol_spec))
inv = json.loads(allocate_weights(json.dumps(inv_vol_spec)))
pd.DataFrame(inv["allocations"])

,id,weight,capital,volatility
0,low_vol_carry,0.896583,896582.95,0.001140
1,trend,0.023224,23224.23,0.044017
2,value,0.080193,80192.82,0.012748


In [7]:
rb_spec = {
    "scheme": "risk_budget",
    "total_capital": 1_000_000.0,
    "strategies": [
        {"id": "equity_sleeve", "risk_budget": 0.50},
        {"id": "credit_sleeve", "risk_budget": 0.30},
        {"id": "rates_sleeve", "risk_budget": 0.20},
    ],
    "covariance": COVARIANCE,
    "money_decimal_places": 2,
}
rb = json.loads(allocate_weights(json.dumps(rb_spec)))
pd.DataFrame(rb["allocations"])

,id,weight,capital,risk_contribution
0,equity_sleeve,0.270475,270474.98,0.5
1,credit_sleeve,0.241456,241456.36,0.3
2,rates_sleeve,0.488069,488068.66,0.2


## Takeaways

- `parametric_var_decomposition_typed` returns an Euler VaR / ES split whose component VaRs sum to the portfolio VaR (residual ~ 0), with optional marginal and incremental VaR.
- `historical_var_decomposition_typed` produces the same split from a positions x scenarios P&L matrix; marginal / incremental are `None` for the historical method.
- `evaluate_risk_budget_typed` flags positions whose component VaR exceeds their target risk share beyond a utilisation threshold.
- `allocate_weights` (with `validate_allocation_json`) sizes strategies under inverse-volatility or risk-budget schemes from a JSON spec.